# CTDenoiser — Colab Setup

Run each cell top-to-bottom. The repo lives on Google Drive so
weights survive session restarts.

**Private repo:** before running, add a Colab **secret** named
`GITHUB_TOKEN` (key icon in the left sidebar) set to a GitHub
personal-access-token with `repo` scope. The clone cell falls back to a
hidden prompt if the secret is missing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess

REPO_SLUG = 'tsereda/CTDenoiser'
BRANCH    = 'claude/init-project-setup-vWHBI'
REPO_ROOT = '/content/drive/MyDrive/CTDenoiser'

# --- GitHub token (private repo) -------------------------------------------
# Preferred: add a Colab secret named GITHUB_TOKEN (key icon in left sidebar),
# value = a PAT with 'repo' scope. Falls back to a hidden prompt.
token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not token:
    import getpass
    token = getpass.getpass('GitHub token (repo scope): ').strip()

auth_url = f'https://{token}@github.com/{REPO_SLUG}.git'

# --- Detect a broken/empty clone and start fresh ---------------------------
is_repo = os.path.isdir(os.path.join(REPO_ROOT, '.git'))
if os.path.isdir(REPO_ROOT) and not is_repo and os.listdir(REPO_ROOT):
    print('Found a non-git directory at REPO_ROOT (failed earlier clone) — removing.')
    shutil.rmtree(REPO_ROOT)
    is_repo = False

if not is_repo:
    os.makedirs(os.path.dirname(REPO_ROOT), exist_ok=True)
    print('Cloning...')
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, auth_url, REPO_ROOT], check=True
    )
else:
    print('Repo exists, updating.')

%cd {REPO_ROOT}
# Keep the token out of .git/config; pass it per-command instead.
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://github.com/{REPO_SLUG}.git'], check=False)
!git -c credential.helper= fetch {auth_url} {BRANCH}
!git checkout {BRANCH}
!git -c credential.helper= pull {auth_url} {BRANCH}
print('\\nHEAD:'); !git log --oneline -1

In [ ]:
%cd /content/drive/MyDrive/CTDenoiser
!pip install -q -r requirements.txt

In [ ]:
%cd /content/drive/MyDrive/CTDenoiser
!pytest -q

In [ ]:
# Smoke run on synthetic data (no real CT needed)
%cd /content/drive/MyDrive/CTDenoiser
!python -m ctdenoiser.train --model ctformer --epochs 1

In [ ]:
# --- Real data (TCIA LDCT-and-Projection-data) ---
# Build ldct_cache.h5 once with your TCIA download/preprocessing notebook
# (keys: <pid>_low / <pid>_full, shape (num_slices, H, W), raw HU).
# Split is by patient; validation uses full-slice overlapped inference.
import os, shutil

DRIVE_CACHE = '/content/drive/MyDrive/ldct_cache.h5'
LOCAL_CACHE = '/content/ldct_cache.h5'

# Local copy = much faster I/O than reading the .h5 off Drive
if os.path.exists(DRIVE_CACHE) and not os.path.exists(LOCAL_CACHE):
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)
    print('Copied cache to local Colab disk.')
assert os.path.exists(LOCAL_CACHE), (
    'ldct_cache.h5 not found - run your TCIA preprocessing notebook first.'
)

# Adjust --epochs / --batch-size to your GPU budget.
%cd /content/drive/MyDrive/CTDenoiser
!python -m ctdenoiser.train \
    --model ctformer \
    --h5-cache /content/ldct_cache.h5 \
    --epochs 50 \
    --batch-size 16 \
    --patch-size 64

# Baseline for comparison:
# !python -m ctdenoiser.train --model redcnn --h5-cache /content/ldct_cache.h5 --epochs 50 --batch-size 16